# OmniCare Clinical Copilot - Notebook 3: Radiology & DICOM Imaging

**Multi-Agent Pipeline:** Load Encounter --> Obtain Medical Images --> Upload to DICOM Store (Healthcare API) --> RadiologyAgent (MedGemma Multimodal) --> Radiology Report --> Firestore Persistence

**Agents used:**
- `RadiologyAgent` — Medical image analysis and radiology report generation (MedGemma multimodal)
- `ClinicalOrchestrator` — Coordinates agents, builds clinical context from prior stages, manages encounter state

**Cloud Services:**
- Google Cloud Healthcare API DICOM Store — Managed DICOMweb storage (replaces self-hosted Orthanc)
- Google Cloud Firestore — Persistent encounter database

**Prerequisites:**
- Run `00_setup_and_models.ipynb` (models loaded, orchestrator initialized)
- Run `01_consultation_audio_soap.ipynb` (consultation SOAP note created)
- Run `02_admission_vitals_fhir.ipynb` (admission note + vitals recorded)
- MedGemma 1.5-4b-it supports image+text input (radiology, dermatology, pathology, ophthalmology)

## 0. Colab Bootstrap (run this first)

Auto-detects environment. In Colab, clones the private repo using your GitHub PAT.

**One-time setup:** In Colab, go to the **Key icon** in the left sidebar > Add a secret named `GITHUB_PAT` with your [GitHub Personal Access Token](https://github.com/settings/tokens).

In [ ]:
# ===========================================================
# Colab Bootstrap - run this cell first (works locally too)
# ===========================================================
import os, sys

try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_DIR = '/content/omnicare-clinical-copilot'

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        token = userdata.get('GITHUB_PAT')
        repo_url = f'https://{token}@github.com/Yashground/omnicare-clinical-copilot.git'
        os.system(f'git clone {repo_url} {REPO_DIR}')
    NOTEBOOKS_DIR = os.path.join(REPO_DIR, 'notebooks')
    os.makedirs('/content/encounters', exist_ok=True)
    os.makedirs('/content/sample_data', exist_ok=True)
else:
    NOTEBOOKS_DIR = os.path.dirname(os.path.abspath('__file__'))

if NOTEBOOKS_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOKS_DIR)

print(f'Environment ready | Colab: {IN_COLAB} | Notebooks dir: {NOTEBOOKS_DIR}')

## 1. Load Encounter from Previous Stages

Load the active encounter created by Notebooks 01-02. The orchestrator automatically
pulls SOAP note and admission data from Firestore to build clinical context for the radiologist.

In [ ]:
import os
import json
import torch
from IPython.display import display, HTML
from PIL import Image

from utils.firestore_db import load_encounter, list_encounters
from utils.dicom_helpers import load_medical_image, download_sample_chest_xray
from utils.healthcare_api import upload_dicom_instance, list_dicom_studies, create_dicom_store

# ---- Verify models & orchestrator from Notebook 0 ----
try:
    _ = medgemma_model.device
    _ = orchestrator.agents
    print(f"MedGemma: loaded on {medgemma_model.device}")
    print(f"Orchestrator: {len(orchestrator.agents)} agents ready")
except NameError:
    print("ERROR: Models or orchestrator not loaded. Run Notebook 0 first!")
    raise

# ---- Load the most recent encounter ----
available = list_encounters()
if not available:
    raise RuntimeError("No encounters found in Firestore. Run Notebooks 01 & 02 first.")

encounter_id = available[-1]
encounter = load_encounter(encounter_id)

# Attach encounter to orchestrator so it can use cached results
orchestrator.encounter_id = encounter_id

# Show encounter summary
patient = encounter.get("patient", {})
consultation = encounter["stages"].get("consultation", {})
admission = encounter["stages"].get("admission", {})

print(f"\nEncounter: {encounter_id}")
print(f"Patient:   {patient.get('name', 'N/A')} (MRN: {patient.get('mrn', 'N/A')})")
print(f"Status:    {encounter.get('status', 'unknown')}")
print(f"SOAP note: {'present' if consultation.get('soap_note') else 'MISSING'}")
print(f"Admission: {'present' if admission.get('admission_note') else 'MISSING'}")

In [ ]:
# Build clinical context from SOAP + admission (same logic the orchestrator uses)
soap = consultation.get("soap_note", {})
admission_note = admission.get("admission_note", "")

parts = []
if soap:
    parts.append(f"Assessment: {soap.get('assessment', 'N/A')}")
    parts.append(f"Plan: {soap.get('plan', 'N/A')}")
if admission_note:
    parts.append(f"Admission: {str(admission_note)[:500]}")

clinical_context = "\n".join(parts) or "No prior clinical context available."

# Seed orchestrator._results so run_radiology can auto-build context
orchestrator._results["consultation"] = {"soap_note": soap}
orchestrator._results["admission"] = {"admission_note": admission_note}

print("Clinical context for radiology (auto-built from prior stages):")
print("-" * 60)
print(clinical_context[:800])

## 2. Obtain Medical Images

Options:
- **Option A:** Download a public sample chest X-ray (NIH ChestX-ray14)
- **Option B:** Use pydicom built-in test DICOM file
- **Option C:** Upload your own DICOM/PNG/JPG files

In [ ]:
%pip install -q pydicom

# Option A: Download a sample chest X-ray (public dataset)
SAMPLE_DIR = "/content/sample_data"
os.makedirs(SAMPLE_DIR, exist_ok=True)

sample_image_path = download_sample_chest_xray(SAMPLE_DIR)
print(f"Sample image: {sample_image_path}")

# Load and display
image, metadata = load_medical_image(sample_image_path)
print(f"\nImage size: {image.size} | Mode: {image.mode}")
print("Metadata:")
for k, v in metadata.items():
    print(f"  {k}: {v}")

display(image.resize((400, 400)))

In [ ]:
# Option B: Use pydicom built-in test DICOM file
test_dicom = None
try:
    import pydicom
    from pydicom.data import get_testfiles_name

    test_dicom = get_testfiles_name("CT_small.dcm")
    ds = pydicom.dcmread(test_dicom)
    print(f"Built-in DICOM test file: {test_dicom}")
    print(f"  Modality:   {getattr(ds, 'Modality', 'N/A')}")
    print(f"  Rows x Cols: {ds.Rows} x {ds.Columns}")
    print(f"  Patient:    {getattr(ds, 'PatientName', 'N/A')}")
except Exception as e:
    print(f"pydicom test file not available: {e}")

In [ ]:
# Option C: Upload your own files (uncomment in Colab)
# from google.colab import files
# uploaded = files.upload()
# custom_image_path = list(uploaded.keys())[0]

# ---- Select which image to use for the rest of the notebook ----
# Change to test_dicom or custom_image_path as needed
image_path = sample_image_path
print(f"Selected image for analysis: {image_path}")

## 3. Upload to DICOM Store (Google Cloud Healthcare API)

Upload the selected image to the managed DICOM store via DICOMweb STOW-RS.
The Healthcare API DICOM store was created in Notebook 00 (`setup_healthcare_stores()`).

**Note:** Only `.dcm` / `.dicom` files can be uploaded to the DICOM store. PNG/JPG images
are analysed locally by MedGemma but skipped for DICOM upload.

In [ ]:
dicom_upload_result = None

if image_path.lower().endswith((".dcm", ".dicom")):
    print(f"Uploading DICOM file to Healthcare API DICOM store...")
    try:
        dicom_upload_result = upload_dicom_instance(image_path)
        print(f"Upload successful: {dicom_upload_result}")
    except Exception as e:
        print(f"DICOM upload failed: {e}")
        print("The RadiologyAgent will still analyse the image locally.")
else:
    print(f"Image is {os.path.splitext(image_path)[1]} format (not DICOM).")
    print("Skipping DICOM store upload. MedGemma will analyse the image directly.")

# List current studies in the DICOM store
print("\nDICOM store studies:")
try:
    studies = list_dicom_studies()
    if studies:
        for i, s in enumerate(studies[:5]):
            study_uid = s.get("0020000D", {}).get("Value", ["N/A"])[0] if isinstance(s, dict) else str(s)
            print(f"  [{i}] Study UID: {study_uid}")
        print(f"  Total studies: {len(studies)}")
    else:
        print("  (no studies in store yet)")
except Exception as e:
    print(f"  Could not list studies: {e}")

## 4. Run Radiology Agent

The `ClinicalOrchestrator.run_radiology()` method:
1. Automatically builds clinical context from the consultation SOAP note and admission note
2. Optionally uploads the image to the Healthcare API DICOM store
3. Runs the `RadiologyAgent` which calls MedGemma multimodal for image analysis
4. Persists the radiology report and image metadata to Firestore

MedGemma 1.5-4b-it is specifically trained on:
- Chest X-rays
- Dermatology images
- Pathology slides
- Ophthalmology (fundus) images

In [ ]:
# Determine modality and body part from image metadata
modality = metadata.get("modality", "X-ray")
body_part = metadata.get("body_part", "Chest")
if modality == "Unknown":
    modality = "X-ray"      # Default for sample chest X-ray PNG
if body_part == "Unknown":
    body_part = "Chest"

print(f"Image:    {os.path.basename(image_path)}")
print(f"Modality: {modality}")
print(f"Body part: {body_part}")
print(f"Upload to DICOM store: {image_path.lower().endswith(('.dcm', '.dicom'))}")

# Run the RadiologyAgent via the orchestrator
# - clinical_context is auto-built from SOAP + admission when left empty
# - upload_to_dicom triggers Healthcare API DICOM upload for .dcm files
radiology_result = orchestrator.run_radiology(
    image_path=image_path,
    modality=modality,
    body_part=body_part,
    upload_to_dicom=True,
)

## 5. Display Radiology Report

The `RadiologyAgent` stores the report in Firestore automatically. Below we display
the structured radiology report along with the analysed image.

In [ ]:
report_text = radiology_result.get("radiology_report", "")
dicom_id = radiology_result.get("dicom_id")

print("=" * 60)
print("RADIOLOGY REPORT")
print("=" * 60)
print(f"Encounter:  {encounter_id}")
print(f"Modality:   {radiology_result.get('modality', modality)}")
print(f"Body Part:  {radiology_result.get('body_part', body_part)}")
print(f"DICOM ID:   {dicom_id or 'local (not uploaded)'}")
print(f"Report len: {len(report_text)} chars")
print("-" * 60)
print(report_text)
print("=" * 60)

# Display the image alongside the report
display(image.resize((400, 400)))

## 6. View Updated Encounter State

The RadiologyAgent has persisted the radiology report, image metadata, and DICOM ID
to Firestore. Load the full encounter to verify all stages.

In [ ]:
# Reload encounter from Firestore to see the updated state
enc = load_encounter(encounter_id)

print(f"Encounter: {encounter_id}")
print(f"Status:    {enc.get('status', 'unknown')}")
print(f"Updated:   {enc.get('updated_at', 'N/A')}")
print()

# Show stage completion summary
for stage_name, stage_data in enc["stages"].items():
    ts = stage_data.get("timestamp")
    status = "COMPLETE" if ts else "pending"
    print(f"  {stage_name:15s} [{status:8s}]  {ts or ''}")

print()

# Show radiology stage details
rad = enc["stages"].get("radiology", {})
images = rad.get("images", [])
reports = rad.get("reports", [])
print(f"Radiology images stored:  {len(images)}")
print(f"Radiology reports stored: {len(reports)}")

if images:
    for i, img in enumerate(images):
        print(f"\n  Image {i+1}:")
        print(f"    Path:     {img.get('path', 'N/A')}")
        print(f"    Modality: {img.get('modality', 'N/A')}")
        print(f"    Body part: {img.get('body_part', 'N/A')}")
        print(f"    DICOM ID: {img.get('dicom_id', 'N/A')}")

In [ ]:
# View RadiologyAgent audit trail from Firestore
try:
    from utils.firestore_db import get_agent_logs
    logs = get_agent_logs(encounter_id, agent_name="radiology_agent")
    if logs:
        print("RadiologyAgent audit trail:")
        for log in logs:
            print(f"  [{log['timestamp']}] {log['action']}: {log['details']}")
    else:
        print("No agent logs found (Firestore logging may not be active).")
except Exception as e:
    print(f"Could not retrieve agent logs: {e}")

In [ ]:
print(f"Radiology stage complete for encounter: {encounter_id}")
print(f"\nNext: Run 04_discharge_summary.ipynb")
print(f"  The DischargeAgent will aggregate SOAP + admission + radiology")
print(f"  to generate the final discharge summary and ICD-10 codes.")